In [ ]:
!pip install scipy scikit-learn
import sys
!{sys.executable} -m pip install scipy scikit-learn

In [16]:


import logging
import random

import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
from sklearn.metrics.pairwise import paired_distances
from sklearn.neighbors import kneighbors_graph

logger = logging.getLogger("districting_tools.horizontal_diffusion_map")


class HorizontalDiffusionMap:
    """Run the horizontal diffusion map algorithm on a fibered dataset.

    Each fibre corresponds to one precinct and contains all district
    realizations associated with that precinct across the ensemble.
    Horizontal diffusion connects (p, A) <-> (p, B): same precinct,
    different district.

    Exposes self.W and self.D_W so that an external combiner can form
    the full transition matrix:

        T = omega * Dv_inv @ Av  +  (1 - omega) * Dw_inv @ W

    Attributes:
        W:   symmetrized, alpha-normalized affinity matrix (sparse CSR)
        D_W: diagonal sparse matrix of row sums of W (for building T)
        evals: eigenvalues of the Markov operator, descending
        evecs: corresponding eigenvectors (in the original basis)
        transition_matrix: row-stochastic Markov matrix P (sparse CSR)
    """

    def __init__(
        self,
        data: list,
        alpha: float,
        bandwidth: float,
        name: str,
        num_evecs: int = 10,
        mode: str = "knn",
        kernel: str = "rbf",
        metric: str = "euclidean",
        **kwargs,
    ):
        """Initialize a HorizontalDiffusionMap from a fibered dataset.

        Args:
            data: list of length num_precincts; data[j] is a np.array of
                  shape (num_districts_containing_j, encoding_length).
                  Each row encodes one district realization for precinct j.
            alpha: density-normalization exponent (matches diffusion_map.py)
            bandwidth: Gaussian kernel bandwidth
            name: ensemble name used in checkpoint filenames
            num_evecs: number of eigenpairs to compute
            mode: neighbourhood mode; currently only "knn" is supported
            kernel: kernel type; currently only "rbf" is supported
            **kwargs: passed through; must include n_neighbors when mode="knn"
        """
        n_neighbors = kwargs.get("n_neighbors", 10)
        self.name = (
            f"hdm_{name}_{alpha}_{num_evecs}_{kernel}_{metric}_{mode}_{n_neighbors}"
        )

        #TODO
        """Iterate over each precinct fibre independently. Horizontal diffusion 
        is strictly confined across a fibre: (p, A) <-> (p, B) — same precinct, different district 
        so we never connect points belonging to different precincts. We cap k at fibre_size - 1 
        to avoid a point becoming its own nearest neighbour, which would pollute distances with zeros. 
        include_self=False for the same reason; self-loops are recovered implicitly when we symmetrize W = (W + W.T) / 2 below. 
        Singleton fibres (only one district realization for a precinct) have no horizontal neighbours and are skipped entirely.
        """

        rows_list = []
        cols_list = []
        vals_list = []
        global_offset = 0

        for fibre_points in data:
            fibre_points = np.asarray(fibre_points, dtype=float)
            fibre_size = len(fibre_points)

            # Singleton fibres have no horizontal neighbours
            if fibre_size <= 1:
                global_offset += fibre_size
                continue

            k = min(n_neighbors, fibre_size - 1)  # exclude self from neighbour count


            # we add self-loops via symmetrization.
            A_local = kneighbors_graph(
                fibre_points,
                n_neighbors=k,
                metric="jaccard",
                include_self=False,
            )

            local_rows = np.repeat(np.arange(A_local.shape[0]), np.diff(A_local.indptr))
            local_cols = A_local.indices


        
            """For each point and each of its k nearest neighbours, the Jaccard distance d(A,B)=∣A△B∣/∣A∪B∣ is transformed into an affinity weight: 
            W_ij = exp(-bandwidth * d(i,j)^2) where d(A, B) = |A Δ B| / |A ∪ B| (symmetric difference over union — the Hamming distance between district encodings). 
            
            NOTE: bandwidth controls the rate of diffusion. 
            bandwidth -> inf : kernel decays sharply; only very close districts have postive value. 
            bandwidth -> 0 : all weights approach 1; random walk reduces to a uniform coin-flip over neighbours. 
            
            Weights are accumulated as (row, col, val) triples rather than a dense matrix; 
            most pairs share no edge, so a dense matrix would be almost entirely zeros. 
            
            It would be cool to later replace Jaccard distance with a function that incorporates partisan data, 
            giving a geometry that reflects political similarity rather than just geographic overlap.
"""


            #   W_ij = exp(-bandwidth * d(i,j)^2)
            X = fibre_points[local_rows].astype(bool)
            Y = fibre_points[local_cols].astype(bool)
            
            intersection = np.logical_and(X, Y).sum(axis=1)
            union = np.logical_or(X, Y).sum(axis=1)
            
            local_distances = 1 - (intersection / union)
            
            local_vals = np.exp(-bandwidth * (local_distances ** 2))

      
            rows_list.append(global_offset + local_rows)
            cols_list.append(global_offset + local_cols)
            vals_list.append(local_vals)

            global_offset += fibre_size

        self.num_points = global_offset


        if rows_list:
            all_rows = np.concatenate(rows_list)
            all_cols = np.concatenate(cols_list)
            all_vals = np.concatenate(vals_list)
        else:
            all_rows = np.array([], dtype=int)
            all_cols = np.array([], dtype=int)
            all_vals = np.array([], dtype=float)

        W = csr_matrix(
            (all_vals, (all_rows, all_cols)),
            shape=(self.num_points, self.num_points),
        )

        W = (W + W.T) / 2

        """Row-normalize W^(alpha) to get a row-stochastic matrix: 
        P = D_a^(-1) W^(alpha)
        """

        d = np.asarray(W.sum(axis=1)).ravel()
        d[d == 0] = 1.0  
        D_inv_a = diags(d ** (-alpha))
        W_a = D_inv_a @ W @ D_inv_a


        d_a = np.asarray(W_a.sum(axis=1)).ravel()
        d_a[d_a == 0] = 1.0  
        D_a_inv = diags(1.0 / d_a)

        P = (D_a_inv @ W_a).tocsr()
        
        """P is not symmetrics so we diagonalize the symmetric conjugate
        M_sym = D_a^(-1/2) W^(alpha) D_a^(-1/2)
        
        which shares all eigenvalues with P.  If M_sym v = lambda v,
        then P w = lambda w  where  w = D_a^(-1/2) v.
        
        Recovering eigenvectors uses elementwise multiplication (*),
        not matrix multiply (@).  D_a_inv_sqrt is a diagonal matrix,
        so * broadcasts it row-wise onto evecs_sym — applying @ would
        treat it as a full matrix product and give the wrong result.
        
        Eigenpairs are sorted in descending order; the leading
        eigenvector (lambda = 1) is trivial, so embed() skips it."""

        D_a_inv_sqrt = diags(d_a ** (-0.5))
        M_sym = D_a_inv_sqrt @ W_a @ D_a_inv_sqrt

        evals, evecs_sym = eigsh(M_sym, k=num_evecs, which="LA")
        evecs = D_a_inv_sqrt * evecs_sym

        order = evals.argsort()[::-1]
        self.evals = evals[order]
        self.evecs = evecs[:, order]


        self.W = W_a                        # alpha-normalized affinity
        self.D_W = diags(d_a)               # row-sum diagonal of W_a
        self.transition_matrix = P          # row-stochastic P (CSR)
#TODO END

    def embed(self, n_dimensions: int = 2, t: float = 0.0) -> np.ndarray:
        """Return the t-step diffusion embedding (skips the trivial eigenvector)."""
        return self.evecs[:, 1 : n_dimensions + 1] * (
            self.evals[1 : n_dimensions + 1] ** t
        )

    def random_walk(self, idx: int = 0):
        """Generate vertices of a random walk starting at idx."""
        while True:
            p = random.uniform(0, 1)

            start = self.transition_matrix.indptr[idx]
            end = self.transition_matrix.indptr[idx + 1]

            logger.debug(
                f"Start random walk at idx {idx}; {end - start} neighbours, "
                f"probabilities {self.transition_matrix.data[start:end]}"
            )

            next_idx = None
            for i in range(start, end):
                p -= self.transition_matrix.data[i]
                if p <= 0:
                    next_idx = self.transition_matrix.indices[i]
                    break

            if next_idx is None and start < end:
                next_idx = self.transition_matrix.indices[end - 1]

            if next_idx is None:
                return

            idx = next_idx
            yield idx

    @staticmethod
    def load(fname):
        raise NotImplementedError

    def save(self, fname):
        raise NotImplementedError



In [17]:
data = [

    # Fibre for precinct 10
    np.array([
        [1,0,1,0,1],
        [1,1,1,0,0],
        [1,0,1,1,0],
    ]),

    # Fibre for precinct 11
    np.array([
        [0,1,1,0,1],
        [0,1,1,1,0],
    ]),

    # Fibre for precinct 12
    np.array([
        [1,1,0,0,1],
        [1,1,0,1,1],
        [1,0,0,1,1],
    ]),
]
hdm = HorizontalDiffusionMap(
    data=data,
    alpha=0.5,
    bandwidth=1.0,
    name="toy_hdm",
    num_evecs=3,
    n_neighbors=2,
)

print("W:")
print(hdm.W.toarray())

print("\nTransition matrix:")
print(hdm.transition_matrix.toarray())

print("\nRow sums:")
print(np.array(hdm.transition_matrix.sum(axis=1)).ravel())

W:
[[0.         0.5        0.5        0.         0.         0.
  0.         0.        ]
 [0.5        0.         0.5        0.         0.         0.
  0.         0.        ]
 [0.5        0.5        0.         0.         0.         0.
  0.         0.        ]
 [0.         0.         0.         0.         1.         0.
  0.         0.        ]
 [0.         0.         0.         1.         0.         0.
  0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.52284709 0.45326185]
 [0.         0.         0.         0.         0.         0.52284709
  0.         0.52284709]
 [0.         0.         0.         0.         0.         0.45326185
  0.52284709 0.        ]]

Transition matrix:
[[0.        0.5       0.5       0.        0.        0.        0.
  0.       ]
 [0.5       0.        0.5       0.        0.        0.        0.
  0.       ]
 [0.5       0.5       0.        0.        0.        0.        0.
  0.       ]
 [0.        0.        0.        0.        1.  

C:\Users\161jc\AppData\Local\uv\cache\builds-v0\.tmpodZRvy\lib\site-packages\sklearn\metrics\pairwise.py:2462: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
C:\Users\161jc\AppData\Local\uv\cache\builds-v0\.tmpodZRvy\lib\site-packages\sklearn\metrics\pairwise.py:2462: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
C:\Users\161jc\AppData\Local\uv\cache\builds-v0\.tmpodZRvy\lib\site-packages\sklearn\metrics\pairwise.py:2462: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


In [10]:
walker = hdm.random_walk(0)

for _ in range(10):
    print(next(walker))

1
0
2
0
1
0
2
0
1
0
